# Fill nans using random forest models

In [ ]:
# Libraries
import os, shutil
import numpy as np
import pandas as pd
import xarray as xr
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from dask_ml.model_selection import GridSearchCV as gscv 
from dask_ml.wrappers import ParallelPostFit
from sklearn.metrics import r2_score, root_mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir04 = '../paper_ecosystem_carbon/output/04_dgvm_prep/'

---

### Pre-processing

In [ ]:
# Get prepared dgvm data
ds_dgvm = xr.open_zarr(f'{dir04}ds_dgvm_limited_r025.zarr')

In [ ]:
# Get regridded covariate datasets
# Soilgrids 2017
ds_soilgrids2017 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2017_r025.zarr')
# Soilgrids 2020
ds_soilgrids2020 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2020_r025.zarr')
# Worldclim
ds_worldclim = xr.open_zarr(f'{dir01}ds_prep_worldclim_r025.zarr')
# Geom90m
ds_geom90m = xr.open_zarr(f'{dir01}ds_prep_geom90m_r025.zarr')

In [ ]:
# Get land mask
ds_land = xr.open_dataset(dir01 + 'ds_luh2_land_mask_r025.nc')

In [ ]:
# List of ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

In [ ]:
# List with covariates
vars_soilgrids2017 = ['soilgrids2017_bdricm', 'soilgrids2017_bdrlog',
                      'soilgrids2017_bdticm']

vars_soilgrids2020 = ['soilgrids2020_cec', 'soilgrids2020_cfvo', 
                      'soilgrids2020_clay', 'soilgrids2020_phh2o', 
                      'soilgrids2020_sand', 'soilgrids2020_silt']

vars_worldclim = ['worldclim_bio1', 'worldclim_bio3', 'worldclim_bio4', 
                  'worldclim_bio5', 'worldclim_bio6', 'worldclim_bio12', 
                  'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15', 
                  'worldclim_elev']

vars_geom90m = ['geom90m_convergence', 'geom90m_cti', 'geom90m_eastness',
               'geom90m_northness', 'geom90m_slope', 'geom90m_spi']

list_var_exp = [*vars_soilgrids2017, *vars_soilgrids2020, *vars_worldclim, 
                *vars_geom90m]

In [ ]:
# List of used models
list_model = ['cablepop', 'classic', 'ibis', 'jsbach', 'jules', 'lpjguess', 
              'lpjml', 'ocn', 'orchidee', 'sdgvm', 'yibs']

In [ ]:
# Prepare covariate dataset
ds_var = xr.merge([ds_soilgrids2017, ds_soilgrids2020, ds_worldclim, 
                   ds_geom90m])
ds_var = ds_var[list_var_exp]

In [ ]:
# Create data
df_base = xr.merge([ds_land, ds_var, ds_dgvm]) \
    .to_dataframe() \
    .reset_index()

# Drop non-land rows
df_base = df_base[df_base.luh2_land_mask == True]

---

### Create random forest models and predict nan values of prepared model output

In [ ]:
# Initialize dask
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(queue='compute',
                       cores=48,
                       memory='256 GB',
                       account='bm0891', 
                       interface="ib0",
                       walltime='04:00:00',
                       local_directory='../dask/',
                       job_extra_directives=[
                           '-o ../dask/LOG_worker_%j.o',
                           '-e ../dask/LOG_worker_%j.o'])

# Scale dask clusters
cluster.scale(jobs=5) #15

# Configurate dashboard url
import dask
dask.config.config.get('distributed').get('dashboard') \
    .update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})

# Create client
from dask.distributed import Client
client = Client(cluster)
client

In [ ]:
def get_df_m(m):
    """Returns dataframe with model data"""
    
    # Dataframe with model data
    df_m = df_base[df_base.model==m]
    return df_m

In [ ]:
def get_df_sel(m, ctype, etype):

    """Returns dataframe with training and testing data of model"""
    
    # Define variable
    v = f'{ctype}_{etype}'
    # Dataframe with model data
    df_m = get_df_m(m)
    # Dataframe with training and testing data
    df_sel = df_m[(df_m[v] == df_m[v])]    
    return df_sel

In [ ]:
def random_forest_params_rank(X_train, y_train):

    """Find 10 best parameters for random forest regression in 
       pre-defined parameter space
    """
    
    # Create pipline for randomforest regressor, needed for gridserachcv
    pipe = make_pipeline(
        # StandardScaler(),
        RandomForestRegressor(),
        )
            
    # Select parameters of grid
    param_grid = dict(
        #randomforestregressor__max_depth = [50],
        randomforestregressor__n_estimators = [100, 200, 300], #[50, 100, 150],
        randomforestregressor__min_samples_leaf = [2, 3, 4, 5],
        randomforestregressor__max_features = [2, 3, 4, 5, 6, 7, 8, 9, 10],
        randomforestregressor__random_state = [42]
        )
        
    # Grid search with cross validation
    grid_results = gscv(pipe, param_grid, 
        cv=5, # five-fold cross validation
        scoring='r2', # r2 as evaluation criteria # sklearn: R2 gives the same ranking as squared error.
        refit=False)
        
    # Fit models
    grid_results_fit = grid_results.fit(X_train, y_train)

    # Extract rank_test_score and parameters
    df_params_rank = pd.DataFrame(grid_results_fit.cv_results_) \
        [
        'rank_test_score',
        'param_randomforestregressor__min_samples_leaf',
        'param_randomforestregressor__max_features',
        'param_randomforestregressor__n_estimators',
        'param_randomforestregressor__random_state',
        ] \
        .sort_values('rank_test_score') \
        .reset_index(drop=True)
        
    # Rename columns
    df_params_rank.columns = [
        i[29:] if i.startswith('param_randomforestregressor__') 
        else i for i in df_params_rank.columns
        ]

    return df_params_rank

In [ ]:
def parallel_postfit_predict(X_train, y_train, X_predict, params):
    """
    Fits a ParallelPostFit RandomForestRegressor model with the given parameters
    and writes predictions to a Parquet file.

    Parameters:
        X_train: Training feature set
        y_train: Training target set
        X_predict: Features to make predictions on
        params: Dictionary of model parameters
        output_path: Path to save the output Parquet file
    """
    # Initialize and fit the model
    rfr = ParallelPostFit(
        estimator=RandomForestRegressor(
            min_samples_leaf=params['min_samples_leaf'],
            max_features=params['max_features'],
            n_estimators=params['n_estimators'],
            random_state=params['random_state'],
            n_jobs=-1
        ),
        scoring='r2'
    )
    rfr.fit(X_train, y_train)

    # Return predictions
    return rfr.predict(X_predict)

In [ ]:
def get_r2_rmse(df_test, dd_rffill, ctype, etype):

    """ Returns R2 values from test and predicted values"""
    
    # Merge testing dataframe and dataframe with predicted values
    df_test2 = df_test[['lat', 'lon', f'{ctype}_{etype}']] \
        .merge(dd_rffill, how='left', on=['lat', 'lon'])

    # define y_true and y_pred
    y_true = df_test2[f'{ctype}_{etype}']
    y_pred = df_test2[f'rffill_{ctype}_{etype}']

    # Get r2 and rmse values
    v_r2 = r2_score(y_true, y_pred)
    v_rmse = root_mean_squared_error(y_true, y_pred)
        
    return round(v_r2, 2), round(v_rmse, 2)

In [ ]:
def rffill(m, ctype, etype):


    """Returns dataarray with predicted values as ensemble mean of 10 best 
    performing models. Includes identification of model parameters of these 
    10 best performing models, predicition of values, and averaging of these 
    predicitions
    
    Adds information to df_rfstats"""
    
    df_m = get_df_m(m)
    df_sel = get_df_sel(m, ctype, etype)
    ncells = len(df_sel)
    
    v = f'{ctype}_{etype}'
    
    # Predict if more than 300 datapoints for training and testing
    if ncells > 300:
    
        # Split dataframe with selected training and testing rows
        df_train = df_sel.sample(frac=0.8, random_state=42)  # 80%
        df_test = df_sel.drop(df_train.index)  # Remaining 20%

        # Number of training and testing grid cells
        ntrain = len(df_train)
        ntest = len(df_test)

        # Define x_train and y_train
        X_train = df_train[list_var_exp]
        y_train = df_train[v]
        
        # Dataframe with data of explanatory variables
        X_predict = df_m[list_var_exp]
    
        # Get list of best performing model parameters
        df_params = random_forest_params_rank(X_train, y_train)
    
        # Create dataframe to store predicted values
        dd_rffill = df_m[['lat', 'lon']].copy()
        
        # Predict values for 10 best-preforming models
        for rank in range(1, 11):
        
            # Filter parameters for the current rank
            df_params_rank = df_params[df_params.rank_test_score == rank]
                
            # Extract parameters as a dictionary
            params = {'min_samples_leaf': df_params_rank.min_samples_leaf.item(),
                      'max_features': df_params_rank.max_features.item(),
                      'n_estimators': df_params_rank.n_estimators.item(),
                      'random_state': df_params_rank.random_state.item()
                     }
            
            # Run the prediction
            rfr_predict = parallel_postfit_predict(X_train, y_train, X_predict, params)
            
            # Add column with predicted values to dataframe df_rfrx
            dd_rffill[f'rfr{rank}'] = rfr_predict
        
        # Add column with predicition mean of 10 best-performing models
        dd_rffill[f'rffill_{v}'] = dd_rffill.filter(regex="^rfr") \
                .mean(axis=1).round(0).astype('int16')
        
        # Keep only lat, lon, and mean 
        dd_rffill = dd_rffill[['lat', 'lon', f'rffill_{v}']]
    
        # Get r2 and rmse values
        v_r2, v_rmse = get_r2_rmse(df_test, dd_rffill, ctype, etype)
        
        # Create dataset from dataframe
        ds_rffill = dd_rffill \
            .set_index(['lat', 'lon']) \
            .to_xarray() \
            .sortby('lat', ascending=False) \
            .fillna(-9999) \
            .astype('int16') \
            .reindex_like(ds_dgvm, method=None, fill_value=-9999)
        
        # Get array
        da_dgvm = ds_dgvm.sel(model=m)[v]
        
        # Change nan value
        da_rffill = ds_rffill.where(ds_rffill != -9999)[f'rffill_{v}']
        
        # Return filled array only if r2 value larger than 0.5
        if v_r2 > 0.5:
            # Fill nan values with predicted rf values
            da_out = xr.where(da_dgvm == da_dgvm, da_dgvm, da_rffill)
        else:
            da_out = xr.full_like(ds_dgvm.sel(model=m)[v], np.nan)
    
    else:
        # If no dgvm values create nan array
        da_out = xr.full_like(ds_dgvm.sel(model=m)[v], np.nan)
        ntrain, ntest, v_r2, v_rmse = np.nan, np.nan, np.nan, np.nan
    
    # Add information to rfstats dataframe
    df_rfstats.loc[len(df_rfstats)] = [m, ctype, etype, ncells, ntrain, ntest, 
                                       v_r2, v_rmse]
    
    # Adjust name and dimensions
    da_out = da_out \
        .rename(f'{ctype}_{etype}') \
        .expand_dims('model') \
        .compute()

    return da_out

---

In [ ]:
%%time
# Loop through every model, ctype, and etype, predict values if enough training data exist
# using ensemble mean of ten best-performing random forest models
# Store output in dataset
# Store stats in dataframe

# Create dataframe for stats
df_rfstats = pd.DataFrame(columns=['m', 'ctype', 'etype', 'nsel', 'ntrain', 'ntest', 'v_r2', 'v_rmse'])

# Loop through all model, ctype, and etype combinations
ds_rffill = xr.Dataset()
for etype in list_etype:
    for ctype in ['cveg', 'csoil']:
        da_rffill = xr.merge([rffill(m, ctype, etype) for m in list_model])
        ds_rffill[f'{ctype}_{etype}'] = da_rffill[f'{ctype}_{etype}']

In [ ]:
# Chunk dataset with predicted values
ds_rffill = ds_rffill.chunk(model=1)

# Fix encoding, otherwise error when export
for var in ds_rffill.variables:
    ds_rffill[var].encoding.clear()
    
# Export dataset
ds_rffill.to_zarr(f'{dir04}ds_dgvm_rffill_r025.zarr', mode='w');

# Export stats dataframe
df_rfstats.to_csv(f'{dir04}df_rf_fillnan_stats.csv', mode='w', index=False)